## Building the chatbot with multiple tools using LANGGRAPH

In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

/var/folders/rc/76flhrzn4pgf33r81dt4lky00000gn/T/ipykernel_3892/4084895585.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
api_arxiv_wrapper=ArxivAPIWrapper(top_k_results=2)
arxiv=ArxivQueryRun(api_wrapper=api_arxiv_wrapper)
print(arxiv.name)

arxiv_tool = ArxivQueryRun(
    api_wrapper=ArxivAPIWrapper()
)

arxiv


In [3]:
import arxiv

search = arxiv.Search(
    query="Attention is all you need",
    max_results=1
)

try:
    paper = next(search.results())
    print(paper.title)
except Exception as e:
    print(type(e), e)

<class 'arxiv.arxiv.HTTPError'> Page request resulted in HTTP 301: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifically, we replace the attention layer in a vision transformer with a feed-forward layer applied over the patch dimension. The resulting architecture is simply a series of feed-forward layers applied over the patch and feature dimensions in an alternating fashion. In experiments on ImageNet, this architecture performs surprisingly well: a ViT/DeiT-base-sized model obtains 74.9\% top-1 accuracy, compared to 77.9\% and 79.9\% for ViT and DeiT respectively. These results indicate that aspects of vision transformers other than attention, such as the patch embedding, may be more responsible f

In [4]:
api_wiki_wrapper=WikipediaAPIWrapper(top_k_results=2)
wiki=WikipediaQueryRun(api_wrapper=api_wiki_wrapper)
wiki.name

'wikipedia'

In [5]:
wiki.invoke("what is machine learning")

'Page: Machine learning\nSummary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.\nStatistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.\nFrom a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.\n\nPage: Grokking (machine learning)\nSumm

In [6]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ['TAVILY_API_KEY']=os.getenv('TAVILY_API_KEY')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily=TavilySearchResults()

/var/folders/rc/76flhrzn4pgf33r81dt4lky00000gn/T/ipykernel_3892/3880125110.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily=TavilySearchResults()


In [8]:
tavily.invoke('provide me the recent AI news for july 15th 2026')

[{'title': 'DX Today AI Daily Brief - Wednesday, July 15, 2026',
  'url': 'https://www.youtube.com/watch?v=epAXKHX5WUo',
  'content': "Today's briefing covers twelve stories from across the AI ecosystem. Google unveils its expanded Gemini Enterprise platform for building and governing fleets of AI agents at Cloud Next. IBM shares plunge after preliminary second quarter results miss expectations as customers shift budgets to AI hardware. Twenty six Meta employees sue over claims that AI software targeted workers with disabilities in layoffs. Reflection AI signs a compute deal worth over one billion dollars with Nebius. Nvidia slashes its authorized AI chip buyer list in Singapore, Malaysia, and Japan to fight diversion to China. New York becomes the first state to freeze new hyperscale data centers under an executive order from Governor Hochul. Reuters reveals xAI installed 59 gas turbines for Colossus 2 without federal [...] company Cribble announced it is acquiring Cardinal Ops, an Is

In [11]:
# combine all the tools in the list

tools=[tavily, wiki, arxiv_tool]

In [12]:
from langchain_groq import ChatGroq

llm=ChatGroq(model='llama-3.1-8b-instant')
llm_tools=llm.bind_tools(tools)

In [13]:
from pprint import pprint

from langchain_core.messages import AIMessage, HumanMessage

llm_tools.invoke([HumanMessage(content=f"What is the recent AI news")])

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'g4741d7zx', 'function': {'arguments': '{"query":"Recent AI news"}', 'name': 'wikipedia'}, 'type': 'function'}, {'id': 'p13vrzdby', 'function': {'arguments': '{"query":"Recent AI news"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 553, 'total_tokens': 588, 'completion_time': 0.0566138, 'completion_tokens_details': None, 'prompt_time': 0.044389725, 'prompt_tokens_details': None, 'queue_time': 0.050322634, 'total_time': 0.101003525}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f66f1-222d-7181-b1d8-a65f3016d442-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Recent AI news'}, 'id': 'g4741d7zx', 'type': 'tool_call'}, {'name': 'tavily_search_results_json', 'args': {'query': '

In [14]:
llm_tools.invoke([HumanMessage(content=f"What is the recent AI news")]).tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'Recent AI news'},
  'id': 'bsn96s4n6',
  'type': 'tool_call'}]